# Burning Cost in 30 Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/burning-cost-examples/blob/main/notebooks/quickstart-colab.ipynb)

Three insurance pricing tools you can try right now.

| Library | Problem it solves |
|---|---|
| `insurance-causal` | Did our rate change actually work, or is correlation fooling us? |
| `insurance-fairness` | Which rating factors act as proxies for protected characteristics? |
| `insurance-monitoring` | Has our book shifted since we built the model? |

**No data required.** Synthetic UK motor data generated below.

**Runtime: 5-8 minutes on Colab free tier.**

---

**The scenario:** you work in a UK motor pricing team. Six months ago you introduced a rate change on a subset of segments. You now need to (1) evaluate the causal effect of that rate change on loss ratio, (2) check your model for proxy discrimination before the next FCA review, and (3) confirm the book hasn't drifted since the model was built.

## 1. Install

In [ ]:
!pip install -q insurance-causal insurance-fairness insurance-monitoring polars catboost
print("Done.")

## 2. Synthetic UK motor data

8,000 policies across 30 segments. At period 7 out of 12, 12 of the 30 segments received a rate change that reduced loss ratio by 3pp on average. The control segments are untouched.

We also create a feature-level snapshot for two time windows — training (early periods) and monitoring (recent periods) — to simulate the distribution shift that happens as a real book evolves.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)
N = 8_000

# --- Policy-level features ---
driver_age       = rng.integers(18, 75, size=N).astype(float)
vehicle_group    = rng.choice([1, 2, 3, 4, 5, 6], size=N, p=[0.10, 0.18, 0.22, 0.25, 0.15, 0.10]).astype(float)
ncd_years        = rng.choice([0, 1, 2, 3, 4, 5], size=N, p=[0.10, 0.12, 0.15, 0.20, 0.22, 0.21]).astype(float)
annual_mileage   = rng.lognormal(mean=9.2, sigma=0.5, size=N).clip(1_000, 50_000)
occupation_group = rng.choice(["professional", "manual", "clerical", "student", "retired"],
                               size=N, p=[0.22, 0.18, 0.30, 0.10, 0.20])
postcode_band    = rng.choice(["A", "B", "C", "D", "E"], size=N, p=[0.15, 0.25, 0.30, 0.20, 0.10])
exposure         = rng.uniform(0.5, 1.0, size=N)

# --- True log-loss-rate ---
log_rate = (
    -2.8
    + 0.010 * np.maximum(0, 25 - driver_age)
    - 0.006 * np.maximum(0, driver_age - 60)
    + 0.07 * vehicle_group
    - 0.06 * ncd_years
    + 0.00010 * annual_mileage
    + np.where(occupation_group == "student", 0.20, 0.0)
    + np.where(occupation_group == "manual",  0.08, 0.0)
    + np.where(occupation_group == "retired", -0.10, 0.0)
    + np.where(postcode_band == "A", 0.18, 0.0)  # postcode_band A = urban, higher risk
    + np.where(postcode_band == "E", -0.12, 0.0)
)

# --- Gender: protected attribute ---
# Gender correlates with occupation_group and vehicle_group (realistic proxy structure).
# Gender has NO direct effect on loss ratio — any apparent difference is mediated
# through occupation and vehicle group. This is the proxy discrimination pattern
# that the fairness audit should detect.
p_female = (
    0.50
    + 0.15 * (occupation_group == "clerical").astype(float)
    + 0.12 * (occupation_group == "professional").astype(float)
    - 0.12 * (occupation_group == "manual").astype(float)
    - 0.05 * (vehicle_group > 4).astype(float)
).clip(0.05, 0.95)
gender = np.where(rng.uniform(size=N) < p_female, "F", "M")

# --- Claim outcomes ---
freq = np.exp(log_rate)
claim_count  = rng.poisson(freq * exposure)
claim_amount = np.where(
    claim_count > 0,
    rng.lognormal(7.8, 0.7, size=N) * claim_count,
    0.0,
)
loss_ratio = np.where(
    exposure > 0,
    (claim_amount / 1000) / exposure,
    0.0,
).clip(0, 5)

# --- Naive model predictions (without causal correction) ---
# Slightly biased: ignores postcode_band and occupation
log_pred_naive = -2.8 + 0.008 * np.maximum(0, 25 - driver_age) + 0.06 * vehicle_group - 0.05 * ncd_years
predicted_freq = np.exp(log_pred_naive + rng.normal(0, 0.05, size=N))

df = pd.DataFrame({
    "driver_age":        driver_age,
    "vehicle_group":     vehicle_group,
    "ncd_years":         ncd_years,
    "annual_mileage":    annual_mileage,
    "occupation_group":  occupation_group,
    "postcode_band":     postcode_band,
    "gender":            gender,
    "exposure":          exposure,
    "claim_count":       claim_count,
    "claim_amount":      claim_amount,
    "loss_ratio":        loss_ratio,
    "predicted_freq":    predicted_freq,
})

# --- Panel data for the rate change evaluation ---
# Use make_rate_change_data from insurance-causal — its DGP satisfies parallel
# trends by construction, which is the correct setup for DiD.
from insurance_causal.rate_change import make_rate_change_data

panel_df = make_rate_change_data(
    n_policies=8_000,
    n_segments=30,
    n_periods=12,
    change_period=7,
    treated_fraction=0.4,
    true_att=-0.03,          # 3pp reduction in loss ratio
    outcome="loss_ratio",
    random_state=42,
)

# --- Feature snapshots for drift monitoring ---
# Simulate two windows: policies 0-4999 (early, training period)
# and policies 5000-7999 (recent, monitoring period with mild shift)
df_early  = df.iloc[:5_000].copy()
df_recent = df.iloc[5_000:].copy()
# Simulate book ageing: drivers 1-2 years older on average in the recent window
df_recent["driver_age"] = (df_recent["driver_age"] + rng.normal(1.5, 0.8, size=len(df_recent))).clip(18, 80)
df_recent["annual_mileage"] = (df_recent["annual_mileage"] * rng.lognormal(0.04, 0.04, size=len(df_recent))).clip(1_000, 50_000)

print(f"Policy data: {len(df):,} rows")
print(f"Panel data:  {len(panel_df):,} policy-periods across {panel_df['segment_id'].nunique()} segments")
print(f"Claim frequency (early window): {df_early['claim_count'].sum() / df_early['exposure'].sum():.4f}")
print(f"Claim frequency (recent window): {df_recent['claim_count'].sum() / df_recent['exposure'].sum():.4f}")
print(f"\nGender split: {(df['gender']=='F').mean():.1%} female / {(df['gender']=='M').mean():.1%} male")

---

## Part 1: Did the rate change work?

`insurance-causal` — causal evaluation of a historical rate change

Six months ago, 12 of your 30 pricing segments received a rate increase. The 
loss ratios in those segments have fallen since. But so have loss ratios in the
control segments, driven by a common trend. Naive before/after comparisons will
overstate the effect of the rate change.

**Difference-in-Differences** compares the change in treated segments to the
change in control segments. The estimand is the ATT: the average causal effect
of the rate change on the segments that received it.

In [ ]:
from insurance_causal.rate_change import RateChangeEvaluator

evaluator = RateChangeEvaluator(
    method="auto",           # auto-selects DiD (control group present) or ITS
    outcome_col="loss_ratio",
    period_col="period",
    treated_col="treated",
    change_period=7,
    exposure_col="exposure",
    unit_col="segment_id",
)

result = evaluator.fit(panel_df).summary()
print(result)

In [ ]:
import matplotlib.pyplot as plt

# Pre/post plot: treated and control average loss ratios over time
# Dashed vertical line marks the rate change period.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

evaluator.plot_pre_post(ax=axes[0], title="Treated vs Control — Observed Loss Ratio")
evaluator.plot_event_study(ax=axes[1], title="Event Study — Period-by-Period DiD Coefficients")

plt.tight_layout()
plt.show()

print(f"\nTrue ATT in DGP:  -0.0300")
print(f"Estimated ATT:     {result.att:+.4f}  (95% CI: [{result.ci_lower:+.4f}, {result.ci_upper:+.4f}])")
print(f"p-value: {result.p_value:.4f}")

# Parallel trends: pre-treatment DiD coefficients should be near zero.
# A large F-statistic here is a red flag — the control group was not trending
# similarly to the treated group before the intervention.
pt = evaluator.parallel_trends_test()
print(f"\nParallel trends joint F-test: p={pt.joint_pt_pvalue:.3f}  "
      f"[{'PASS' if pt.passes else 'FAIL'}]")

**Reading the output:**

The ATT estimate should be close to -0.03 (the true effect we planted in the data). 
The event study plot shows the pre-treatment coefficients — if parallel trends holds,
these should all be near zero. The post-treatment coefficients show the treatment effect
over time.

The parallel trends p-value tests whether the pre-treatment event study coefficients
are jointly zero. A large p-value (pass) supports the DiD identifying assumption.

---

## Part 2: Are our rating factors acting as proxies for gender?

`insurance-fairness` — proxy discrimination audit

Since the EU Gender Directive (2012) and FCA Consumer Duty guidance, gender cannot be used
as a direct rating factor. But rating factors that correlate with gender
— occupation, vehicle group, postcode — can produce discriminatory outcomes
indirectly. This is proxy discrimination.

The `detect_proxies` function measures how much information each rating factor
carries about the protected characteristic, using three complementary methods:

- **Proxy R-squared**: how well a CatBoost model can predict gender from the factor alone
- **Mutual information**: non-parametric association, captures non-linear relationships
- **Partial correlation**: linear association after controlling for other factors

We planted a correlation between occupation/vehicle group and gender in the data.
These should be flagged.

In [ ]:
from insurance_fairness import detect_proxies

# detect_proxies expects a Polars DataFrame
df_pl = pl.from_pandas(df)

# Encode gender as binary 0/1 for proxy detection
df_pl = df_pl.with_columns(
    pl.when(pl.col("gender") == "F").then(1).otherwise(0).alias("gender_binary")
)

result_proxy = detect_proxies(
    df=df_pl,
    protected_col="gender_binary",
    factor_cols=["occupation_group", "vehicle_group", "postcode_band",
                 "driver_age", "ncd_years", "annual_mileage"],
    is_binary_protected=True,
    run_proxy_r2=True,
    run_mutual_info=True,
    run_partial_corr=True,
    run_shap=False,          # skipping SHAP — requires a fitted model
    catboost_iterations=100, # quick for demo purposes
)

print(f"Proxy detection for protected attribute: gender")
print(f"Flagged factors (amber or red): {result_proxy.flagged_factors}")
print()
print(result_proxy.to_polars())

In [ ]:
from insurance_fairness import (
    calibration_by_group,
    demographic_parity_ratio,
)

# Calibration by group: is the model equally well-calibrated for male and female policyholders?
# A/E ratio by gender — if these differ, the model is systematically over/underpredicting
# for one group even after accounting for all rating factors.
actual   = df_pl["claim_count"].to_numpy()
predicted = df_pl["predicted_freq"].to_numpy()
exposure_arr = df_pl["exposure"].to_numpy()
gender_arr = df_pl["gender_binary"].to_numpy()

cal_by_group = calibration_by_group(
    actual=actual,
    predicted=predicted,
    groups=gender_arr,
    exposure=exposure_arr,
)
print("Calibration by gender (A/E ratio):")
print(cal_by_group)
print()

# Demographic parity ratio: ratio of mean predicted premium between groups.
# 1.0 = parity. FCA does not mandate parity, but values far from 1.0 warrant
# explanation in your Consumer Duty evidence pack.
dpr = demographic_parity_ratio(
    predicted=predicted,
    groups=gender_arr,
)
print(f"Demographic parity ratio (predicted frequency, F vs M): {dpr:.3f}")
print("  Values near 1.0 indicate similar average predictions across groups.")

In [ ]:
# Visualise proxy scores
import matplotlib.pyplot as plt

scores_df = result_proxy.to_polars().to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Proxy R-squared
colors_r2 = ["#d62728" if r == "red" else "#ff7f0e" if r == "amber" else "#2ca02c"
             for r in scores_df["rag"]]
axes[0].barh(scores_df["factor"], scores_df["proxy_r2"].fillna(0), color=colors_r2)
axes[0].axvline(0.10, color="orange", linestyle="--", linewidth=1, label="amber threshold")
axes[0].axvline(0.25, color="red",    linestyle="--", linewidth=1, label="red threshold")
axes[0].set_xlabel("Proxy R-squared")
axes[0].set_title("How well does each factor predict gender?")
axes[0].legend(fontsize=8)

# Mutual information
axes[1].barh(scores_df["factor"], scores_df["mutual_information"].fillna(0), color=colors_r2)
axes[1].set_xlabel("Mutual information (nats)")
axes[1].set_title("Non-parametric association with gender")

plt.suptitle("Proxy discrimination audit — rating factors vs gender", y=1.02)
plt.tight_layout()
plt.show()

print("\nFactors flagged for further review:")
for f in result_proxy.flagged_factors:
    row = scores_df[scores_df["factor"] == f].iloc[0]
    print(f"  {f}: proxy_r2={row['proxy_r2']:.3f}, MI={row['mutual_information']:.3f}, status={row['rag']}")

**Reading the output:**

`occupation_group` and `vehicle_group` should flag amber or red — we planted a
correlation between these factors and gender in the data. `postcode_band` may also
flag depending on random seed.

Flagging does not mean the factor is impermissible. It means the factor carries
gender-related information and warrants a deeper justification in your Consumer
Duty evidence pack: does the factor predict risk independently of gender?
Is the discriminatory effect proportionate to the legitimate pricing objective?

Factors near zero (driver age, ncd_years) carry little information about gender
in this dataset — they are unlikely to produce indirect discrimination.

---

## Part 3: Has the book drifted?

`insurance-monitoring` — PSI/CSI drift detection and model calibration check

We have two windows: the early period the model was built on, and a more recent
period where drivers are slightly older and higher-mileage. Three questions:

1. **Which features have drifted?** (PSI/CSI)
2. **Is the model still calibrated?** (A/E ratio with Poisson CI)
3. **Has model discrimination degraded?** (Gini drift test)

The `MonitoringReport` runs all three and gives a traffic-light recommendation.

In [ ]:
from insurance_monitoring.drift import psi, csi, wasserstein_distance

# Quick PSI check on the features we expect to have drifted
print(f"{'Feature':<22} {'PSI':>8} {'Band':>8} {'Wasserstein':>14}")
print("-" * 58)

features_to_check = ["driver_age", "annual_mileage", "ncd_years", "vehicle_group"]

for feat in features_to_check:
    ref = df_early[feat].values
    cur = df_recent[feat].values
    p = psi(ref, cur, n_bins=10)
    w = wasserstein_distance(ref, cur)
    band = "green" if p < 0.10 else ("amber" if p < 0.25 else "red")
    print(f"{feat:<22} {p:>8.4f} {band:>8} {w:>14.3f}")

print()
w_age = wasserstein_distance(df_early["driver_age"].values, df_recent["driver_age"].values)
print(f"driver_age has shifted by {w_age:.1f} years on average (in original units).")

In [ ]:
# CSI heatmap: all features in one call
feat_early  = pl.from_pandas(df_early[["driver_age", "annual_mileage", "ncd_years", "vehicle_group"]])
feat_recent = pl.from_pandas(df_recent[["driver_age", "annual_mileage", "ncd_years", "vehicle_group"]])

csi_df = csi(
    reference_df=feat_early,
    current_df=feat_recent,
    features=["driver_age", "annual_mileage", "ncd_years", "vehicle_group"],
)

print("CSI summary (PSI applied per feature):")
print(csi_df)

In [ ]:
from insurance_monitoring.calibration import ae_ratio, ae_ratio_ci

# A/E ratio: are actuals tracking predictions in the recent period?
actual_recent    = df_recent["claim_count"].values
predicted_recent = df_recent["predicted_freq"].values
exposure_recent  = df_recent["exposure"].values

# With Poisson confidence interval (exact Garwood interval for count data)
ae_ci = ae_ratio_ci(
    actual=actual_recent,
    predicted=predicted_recent,
    exposure=exposure_recent,
    alpha=0.05,
    method="poisson",
)

print(f"A/E ratio (recent period): {ae_ci['ae']:.4f}")
print(f"95% CI (Poisson exact):    [{ae_ci['lower']:.4f}, {ae_ci['upper']:.4f}]")
print(f"Observed claims:           {ae_ci['n_claims']:.0f}")
print(f"Expected claims:           {ae_ci['n_expected']:.1f}")
print()

# Segmented A/E by driver age band — see where miscalibration is concentrated
age_band = np.where(df_recent["driver_age"] < 25, "17-24",
           np.where(df_recent["driver_age"] < 40, "25-39",
           np.where(df_recent["driver_age"] < 60, "40-59", "60+")))

ae_seg = ae_ratio(
    actual=actual_recent,
    predicted=predicted_recent,
    exposure=exposure_recent,
    segments=age_band,
)

print("A/E ratio by driver age band:")
print(ae_seg)

In [ ]:
from insurance_monitoring import MonitoringReport

# Full quarterly monitoring report: drift + calibration + discrimination in one call
actual_early    = df_early["claim_count"].values
predicted_early = df_early["predicted_freq"].values
exposure_early  = df_early["exposure"].values

report = MonitoringReport(
    reference_actual=actual_early,
    reference_predicted=predicted_early * exposure_early,  # expected counts
    current_actual=actual_recent,
    current_predicted=predicted_recent * exposure_recent,  # expected counts
    exposure=exposure_recent,
    reference_exposure=exposure_early,
    feature_df_reference=feat_early,
    feature_df_current=feat_recent,
    features=["driver_age", "annual_mileage", "ncd_years", "vehicle_group"],
    murphy_distribution="poisson",  # Murphy decomposition: RECALIBRATE vs REFIT
    n_bootstrap=150,
)

print(f"Monitoring recommendation: {report.recommendation}")
print()

In [ ]:
# Traffic-light summary
ae = report.results_["ae_ratio"]
print(f"A/E ratio: {ae['value']:.3f}  [{ae['band'].upper()}]")
print(f"  95% CI:  [{ae['lower_ci']:.3f}, {ae['upper_ci']:.3f}]")
print()

gini = report.results_["gini"]
print(f"Gini (reference): {gini['reference']:.3f}")
print(f"Gini (current):   {gini['current']:.3f}  [{gini['band'].upper()}]")
print(f"  Change: {gini['change']:+.3f}   p={gini['p_value']:.3f}")
print()

if report.murphy_available:
    m = report.results_["murphy"]
    print(f"Murphy decomposition ({m['verdict']}):")
    print(f"  Discrimination: {m['discrimination_pct']:.1f}% of total error  "
          f"({'stable' if m['discrimination_pct'] > 50 else 'degraded'})")
    print(f"  Miscalibration: {m['miscalibration_pct']:.1f}% of total error")
    print()

print(f"Worst drifting feature: {report.results_['max_csi']['worst_feature']}  "
      f"(CSI={report.results_['max_csi']['value']:.3f}, "
      f"{report.results_['max_csi']['band'].upper()})")
print()
print(f"Recommendation: {report.recommendation}")

In [ ]:
# Full flat table — suitable for writing to Delta or logging to MLflow
print("Complete monitoring results:")
report.to_polars()

In [ ]:
# PSI bar chart: visualise drift across features
import matplotlib.pyplot as plt

csi_results = report.results_["csi"]
features_plot = [r["feature"] for r in csi_results]
csi_vals      = [r["csi"]     for r in csi_results]
bands         = [r["band"]    for r in csi_results]

bar_colors = ["#d62728" if b == "red" else "#ff7f0e" if b == "amber" else "#2ca02c"
              for b in bands]

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(features_plot, csi_vals, color=bar_colors)
ax.axhline(0.10, color="orange", linestyle="--", linewidth=1, label="amber (0.10)")
ax.axhline(0.25, color="red",    linestyle="--", linewidth=1, label="red (0.25)")
ax.set_ylabel("CSI (PSI per feature)")
ax.set_title("Feature stability — training vs monitoring period")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Reading the output:**

`driver_age` and `annual_mileage` should show amber or red CSI — we simulated
book ageing in those dimensions. The stable features (`ncd_years`, `vehicle_group`)
should stay green.

The Murphy decomposition tells you whether any A/E deterioration is a global
calibration issue (fixable with a simple intercept adjustment) or a discrimination
issue (model has lost its ability to rank-order risk — requires a rebuild).

The recommendation follows the arXiv 2510.04556 decision tree:
- `NO_ACTION`: everything stable
- `RECALIBRATE`: A/E has drifted but Gini is stable — update the offset
- `REFIT`: Gini has degraded — the model needs to be rebuilt on recent data
- `MONITOR_CLOSELY`: amber signals — watch the trend next quarter

---

## Next steps

| Library | PyPI | GitHub | Documentation |
|---|---|---|---|
| `insurance-causal` | [`pip install insurance-causal`](https://pypi.org/project/insurance-causal/) | [burning-cost/insurance-causal](https://github.com/burning-cost/insurance-causal) | Causal forests, AutoDML, elasticity, staggered DiD |
| `insurance-fairness` | [`pip install insurance-fairness`](https://pypi.org/project/insurance-fairness/) | [burning-cost/insurance-fairness](https://github.com/burning-cost/insurance-fairness) | Discrimination-free pricing, marginal fairness, DoubleFairness |
| `insurance-monitoring` | [`pip install insurance-monitoring`](https://pypi.org/project/insurance-monitoring/) | [burning-cost/insurance-monitoring](https://github.com/burning-cost/insurance-monitoring) | TRIPODD attribution, PIT sequential monitoring, GiniDriftBootstrap |

**More examples:** [github.com/burning-cost/burning-cost-examples](https://github.com/burning-cost/burning-cost-examples)

**34 libraries covering the full UK pricing workflow:** GLMs, credibility, GAMs,
survival models, GBM-to-GLM distillation, optimal transport pricing, telematics,
conformal prediction intervals, and more. All open-source, all on PyPI.